In [1]:
import pandas as pd

import geopandas as gp
import folium as f
from shapely import wkt
from shapely.geometry import LineString

import re

import ast

### RECON PROJECTS

In [2]:
recon = pd.read_csv("Street_and_Highway_Capital_Reconstruction_Projects_-_Intersection_20260930.csv", header=0)

# filters
#recon = recon[recon["OnStreetName"].str.contains("EXPWY|EXPRESSWAY|EXP|PKWY|PARKWAY|PKY", regex=True, case=False)]

In [3]:
display(recon[recon["OnStreetName"].str.contains("EXP|EXPRESSWAY", regex=True, case=False)][["OnStreetName"]])

,OnStreetName
761,GOWANUS EXPRESSWAY EXIT 19 WB
819,BROOKLYN QUEENS EXPRESSWAY
926,BROOKLYN QUEENS EXPWY ET 27 WB
2890,CROSS BRONX EXPWY NB ET 1 B
2976,STATEN ISLAND EXPWY ET 12 WB
4000,BRUCKNER EXPWY EB EN E 138 ST
4015,BRUCKNER EXPRESSWAY
4034,BRUCKNER EXPRESSWAY EXIT 47 WB
4520,BROOKLYN QUEENS EXPRESSWAY
4735,VAN WYCK EXPWY SB EN LINDEN BLVD


### ROUTES

In [4]:
#read data set
data = pd.read_csv("NYC_Roadway_Jurisdiction.csv", header=0, index_col=0)
#inv =  inv[inv["County_Name"].str.contains("Bronx|Kings|New York|Richmond|Queens", regex=True)]
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 23177 entries, 0 to 23176
Data columns (total 61 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   OBJECTID                       23177 non-null  int64  
 1   Id                             23177 non-null  str    
 2   ROUTE_ID                       23177 non-null  int64  
 3   FROM_MEASURE                   23177 non-null  float64
 4   TO_MEASURE                     23177 non-null  float64
 5   Route_Display_Value            1018 non-null   str    
 6   Route_Signing                  775 non-null    str    
 7   Route_Number                   1017 non-null   float64
 8   Route_Suffix                   311 non-null    str    
 9   Route_Qualifier                21851 non-null  str    
 10  Roadway_Type                   23177 non-null  str    
 11  Roadway_Feature                1109 non-null   str    
 12  Direction_Code                 23177 non-null  int64  
 1

#### Parks

In [5]:
## all aprks
# all parkways prefix
park_name = ["Bay","Belt", "Bronx River", "Cross Island","Eastern", "FDR", "Franklin D. Roosevelt East River" 
         "Grand Central", "Henry Hudson", "Hutchinson River", 
         "Harlem River", "Jackie Robinson", "JFK",
         "Korean War Veterans", "Little Neck","Mosholu", "Northern State", "Ocean", "Prospect", "Pelham",
         "Palisades Interstate", "Rockaway", "Saw Mill River", "Southern State"]

exception = "FDR"

parks_base = [x.upper() for x in park_name]

#pattern = ' PK|'.join(parks_upper)
pattern = r'(?:' + '|'.join(re.escape(name) + r".*\b(?:PKWY|PARKWAY|PKY)\b" for name in park_name if name !=exception) + r"|" + re.escape(exception) + r")"
mask1 = data['Roadway_Name'].apply(lambda x: x.upper() if isinstance(x, str) else x).str.contains(pattern, regex=True,case=False)
mask2 = data["Roadway_Name"].apply(lambda x: x.upper() if isinstance(x, str) else x).str.contains(r"\b(?:PKWY|PARKWAY|PKY)\b", case=False, na=False, regex=True)

mask = mask1 | mask2
pkwy = data.loc[mask].reset_index(drop=True).copy()



In [6]:
parks = pkwy.copy()
parks["geometry"] = parks["geometry"].apply(lambda x: LineString(ast.literal_eval(x)["paths"][0]))


all_parks = gp.GeoDataFrame(
    parks,
    geometry="geometry",
    crs="EPSG:26918"
)

all_parks_crs = all_parks.to_crs("EPSG:4326")


In [7]:
all_parks_crs.crs

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

#### Expressways

In [8]:
# expressways names
#Harlem River Drive

exp = ["Brooklyn–Queens", "Bruckner", 
       "Clearview", "Clove Lakes", "Cross Bronx", 
       "MLK", "Dr. Martin Luther King Jr.", 
       "George Washington Bridge", "Gowanus", 
       "Horace Harding", "Holland Tunnel", 
       "JFK", 
       "Lincoln Tunnel", "Long Island", 
       "Major Deegan",
       "Nassau", "New England Thruway", 
       "Prospect", 
       "Queens-Midtown", 
       "Sheridan", "Staten Island", "Sunrise Highway" 
       "Trans-Manhattan", "Throgs Neck",
       "Van Wyck", 
       "West Shore", "West Side", "Joe DiMaggio", "Westway", "Whitestone", "Willowbrook"]

exception = "Interstate"

exp_base = [x.upper() for x in exp]

#pattern = ' PK|'.join(parks_upper)
pattern_e = r'(?:' + '|'.join(re.escape(name) + r".*\b(?:EXPY|EXPRESSWAY|EXP)\b" for name in exp if name !=exception) + r"|" + re.escape(exception) + r")"
mask1 = data['Roadway_Name'].apply(lambda x: x.upper() if isinstance(x, str) else x).str.contains(pattern_e, regex=True,case=False)
mask2 = data["Roadway_Name"].apply(lambda x: x.upper() if isinstance(x, str) else x).str.contains(r"\b(?:EXPY|EXPRESSWAY|EXP)\b", case=False, na=False, regex=True)

mask = mask1 | mask2
expy = data.loc[mask].reset_index(drop=True).copy()

# expy[["Roadway_Name", "Owning_Jurisdiction_Desc"]]

express_geom = expy.copy()
express_geom["geometry"] = express_geom["geometry"].apply(lambda x: LineString(ast.literal_eval(x)["paths"][0]))


all_exp = gp.GeoDataFrame(
    express_geom,
    geometry="geometry",
    crs="EPSG:26918"
)

all_exp_crs = all_exp.to_crs("EPSG:4326")

In [9]:
display(expy[["Maintenance_Jurisdiction_Desc"]].value_counts())
display(expy[["Owning_Jurisdiction_Desc"]].value_counts())

Maintenance_Jurisdiction_Desc
NYSDOT                           432
City or village                   64
Other Toll Authority              29
Public - Unclaimed                11
Other local agencies               3
Other Public Instrumentality       2
NYS Thruway                        1
Name: count, dtype: int64

Owning_Jurisdiction_Desc    
NYSDOT                          432
City or village                  64
Other Toll Authority             29
Public - Unclaimed               11
Other local agencies              3
Other Public Instrumentality      2
NYS Thruway                       1
Name: count, dtype: int64

In [10]:
## city owned/maintained
city_parks = all_parks_crs[(all_parks_crs["Maintenance_Jurisdiction_Desc"] == "City or village") & (all_parks_crs["Owning_Jurisdiction_Desc"] == "City or village")]
city_parks

## city owned/maintained
city_exp = all_exp_crs[(all_exp_crs["Maintenance_Jurisdiction_Desc"] == "City or village") & (all_exp["Owning_Jurisdiction_Desc"] == "City or village")]
city_exp

##

,OBJECTID,Id,ROUTE_ID,FROM_MEASURE,TO_MEASURE,Route_Display_Value,Route_Signing,Route_Number,Route_Suffix,Route_Qualifier,...,Nat_Highway_Freight_Net_Code,Nat_Highway_Freight_Net_Desc,STRAHNET_Code,STRAHNET_Desc,AADT_Single_Unit,AADT_Combination_Unit,Overlap_Number,SHAPE.STLength(),_length,geometry
6,16842,0dfcd93a-8a17-417e-a0c8-c3e8e3875a18,256012011,0.40,0.622331,NaN,NaN,NaN,NaN,No qualifier,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,357.808476,NaN,"LINESTRING (-73.80995 40.81861, -73.8069 40.81..."
26,62235,34385dde-5279-42a8-851f-2e21c810bb60,311737011,0.00,0.218365,NaN,NaN,NaN,NaN,No qualifier,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,351.425903,NaN,"LINESTRING (-73.88391 40.72804, -73.88375 40.7..."
41,40095,21869d14-87e1-405f-afde-afb2c0b85672,100423021,12.27,12.610000,I495,I,495.0,NaN,No qualifier,...,NaN,NaN,1.0,STRAHNET,NaN,NaN,NaN,547.178079,NaN,"LINESTRING (-73.74381 40.75394, -73.74377 40.7..."
45,48102,2857fa86-7cb9-476a-9659-bd8ffbd33834,279165011,0.00,0.855786,NaN,NaN,NaN,NaN,No qualifier,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1377.256360,NaN,"LINESTRING (-73.82384 40.83434, -73.82377 40.8..."
50,1061,00ce0a14-2a53-438b-8944-35e5453b6493,312525203,0.00,0.174517,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,280.858144,NaN,"LINESTRING (-73.85911 40.73587, -73.85929 40.7..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
528,299492,fc0836d0-a026-4885-a004-206ccfe8f167,278590011,0.65,0.990000,NaN,NaN,NaN,NaN,No qualifier,...,NaN,NaN,NaN,NaN,316.0,34.0,NaN,547.178125,NaN,"LINESTRING (-73.82528 40.82452, -73.8252 40.82..."
529,299666,fc331635-bf46-496b-a707-b666c0eda838,321559011,0.00,0.024885,NaN,NaN,NaN,NaN,No qualifier,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.047923,NaN,"LINESTRING (-73.9207 40.84523, -73.92112 40.8454)"
533,303898,ffcae951-1a2f-42d6-b60b-f1575d502ef6,312525283,0.00,0.116770,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,187.923333,NaN,"LINESTRING (-73.91027 40.72921, -73.91022 40.7..."
536,280892,ec46c661-78aa-4657-851f-1314c7d0f8f9,257281011,0.72,0.914280,NaN,NaN,NaN,NaN,No qualifier,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,312.663147,NaN,"LINESTRING (-73.89999 40.7577, -73.90002 40.75..."


In [11]:
city_exp["Roadway_Name"].unique()

<StringArray>
[                                      'THROGS NECK EXPY EXT',
                                        'QUEENS MIDTOWN EXPY',
                                            'LONG ISLAND EXP',
                                        'THROGS NECK EXPY SR',
     'SLIP RAMP FROM HORACE HARDING EXPY TO RAMP FOR I495 WB',
          'SLIP RAMP FROM QUEENS MIDTOWN EXPY TO CLINTON AVE',
              'SLIP RAMP FROM E 177TH ST TO CROSS BRONX EXPY',
          'SLIP RAMP FROM BROOKLYN QUEENS EXPY W TO 30TH AVE',
                  'SLIP RAMP FROM CLEARVIEW EXPY TO 204TH ST',
                                 'WHITESTONE EXPY TURNAROUND',
         'SLIP RAMP FOR QUEENS MIDTOWN EXPY TO RAMP FOR I495',
              'SLIP RAMP FROM 80TH ST TO QUEENS MIDTOWN EXPY',
                                      'WHITESTONE EXPY SR NB',
                                  'HORRACE HARDING EXPY EAST',
     'SLIP RAMP FROM THROGS NECK EXPRESSWAY TO RAMP FOR I295',
                                       'C

### MAPS

In [12]:
# base map
k = f.Map(tiles='openstreetmap',
          zoom_start=10, max_zoom=23, control_scale=True, location=[40.7128, -74.0060])

In [13]:
# create buffer
# parks
city_projects = gp.GeoDataFrame(recon,
                                geometry=gp.points_from_xy(recon["Longitude"], recon["Latitude"]),
                                crs = "EPSG:4326")

c_project = city_projects.to_crs("EPSG:2663") #pROJECTS
city_park_route = city_parks.to_crs("EPSG:2663") #ROUTES

buffer_distance = 10
park_route_buffer = city_park_route.copy()
park_route_buffer["geometry"] = park_route_buffer.geometry.buffer(buffer_distance)

city_projects_nearby = gp.sjoin(c_project,
                         park_route_buffer,
                         how="inner",
                         predicate="within"
                        )
# Exp
city_exp_route = city_exp.to_crs("EPSG:2663") #ROUTES

exp_route_buffer = city_exp_route.copy()
exp_route_buffer["geometry"] = exp_route_buffer.geometry.buffer(buffer_distance)

city_Exp_projects_nearby = gp.sjoin(c_project,
                         exp_route_buffer,
                         how="inner",
                         predicate="within"
                        )


In [14]:
#Parkway buffer
#BACK TO EPSG:4326

# city_park_route = city_parks.to_crs("EPSG:4326") #ROUTES
# city_projects_nearby = city_projects_nearby.to_crs("EPSG:4326") #pROJECTS
city_park_route = city_parks.copy()
city_projects_nearby = city_projects_nearby.copy()

nearby_layer = f.FeatureGroup(name="Within 10m of routes - Parks", show=True)
for _,row in city_projects_nearby.iterrows():
    f.CircleMarker(location=[row["Latitude"], row["Longitude"]],
                radius=0.5,
                color="#37EC12DA",
                fill=True,
                fill_color="#37EC12DA",
                fill_opacity=0.8,
                zIndex=100,
                popup=f"Cost: {row["ProjectCost"]}\n, Project Type: {row["ProjectType"]}\n, ConstructionEndDate:{ row["ConstructionEndDate"]}\n, Street:{ row["OnStreetName"]}").add_to(nearby_layer)


nearby_layer.add_to(k)


In [15]:
#Exp buffer
#BACK TO EPSG:4326
# city_exp_route = city_exp.to_crs("EPSG:4326") #ROUTES
# city_Exp_projects_nearby = city_Exp_projects_nearby.to_crs("EPSG:4326") #pROJECTS

city_exp_route = city_exp.copy() #ROUTES
city_Exp_projects_nearby = city_Exp_projects_nearby.copy() #pROJECTS

nearby_layer_exp = f.FeatureGroup(name="Within 10m of routes - Express", show=True)
for _,row in city_Exp_projects_nearby.iterrows():
    f.CircleMarker(location=[row["Latitude"], row["Longitude"]],
                radius=0.5,
                color="#FF1900FF",
                fill=True,
                fill_color="#916762AD",
                fill_opacity=0.8,
                zIndex=100,
                popup=f"Cost: {row["ProjectCost"]}\n, Project Type: {row["ProjectType"]}\n, ConstructionEndDate:{ row["ConstructionEndDate"]}\n, Street:{ row["OnStreetName"]}").add_to(nearby_layer_exp)


nearby_layer_exp.add_to(k)

In [16]:
# PARKWAY
f.GeoJson(
    all_parks_crs[["geometry"]].to_json(),
    name="All Parkways Routes",
    style_function=lambda feature: {
        "color": "#B11FEB",
        "Weight": 3,
        "opacity": 1
    }
).add_to(k)

f.GeoJson(
    city_parks[["geometry"]].to_json(),
    name="Parkway Routes (City Owned)",
    style_function=lambda feature: {
        "color": "#FF000000",
        "Weight": 3,
        "opacity": 1
    }
).add_to(k)

#EXPRESSWAY
f.GeoJson(
    all_exp_crs[["geometry"]].to_json(),
    name="All Express Routes",
    style_function=lambda feature: {
        "color": "#03B0B6",
        "Weight": 3,
        "opacity": 1
    }
).add_to(k)

f.GeoJson(
    city_exp[["geometry"]].to_json(),
    name="Expressway Routes (City Owned)",
    style_function=lambda feature: {
        "color": "#341DFF4B",
        "Weight": 3,
        "opacity": 1
    }
).add_to(k)


In [17]:
# ADD ALL PROJECTS
project = f.FeatureGroup(name="Capital Project")

for _,row in recon.iterrows():
    f.CircleMarker(location=[row["Latitude"], row["Longitude"]],
                radius=0.5,
                color="#020202DA",
                fill=True,
                fill_color="#FD2600DA",
                fill_opacity=0.8,
                zIndex=100,
                popup=f"Cost: {row["ProjectCost"]}\n, Project Type: {row["ProjectType"]}\n, ConstructionEndDate:{ row["ConstructionEndDate"]}\n, Street:{ row["OnStreetName"]}").add_to(project)


project.add_to(k)

In [18]:
f.LayerControl().add_to(k)

In [19]:
greyscale_css = """ 
<style>
.leaflet-tile {
    filter: grayscale(100%);
}
</style>
"""

k.get_root().header.add_child(
    f.Element(greyscale_css)
)

In [20]:
k

In [21]:
k.save("map_grayscale.html")